# Customer Shopping Behavior Analysis

## Business Problem

Retail businesses generate large amounts of customer transaction data, but raw data alone does not provide actionable insights.

The goal of this analysis is to understand customer purchasing patterns, identify factors influencing spending behavior, and uncover opportunities to improve revenue, customer retention, and marketing effectiveness.

This analysis explores demographic trends, purchasing habits, subscription behavior, seasonal patterns, and product performance.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('customer_shopping_behavior.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Summary statistics using .describe() without include all it only works for numerical columns
df.describe(include='all')

In [ ]:
# Checking if missing data or null values are present in the dataset

df.isnull().sum()

## Handling Missing Review Ratings

I chose a category-wise median instead of an overall median because customer ratings can vary significantly across product categories. For example, Clothing products may naturally receive different ratings than Accessories.

Using the median within each category preserves category-specific behavior while reducing the influence of extreme ratings.

In [ ]:
df['Review Rating'] = (
    df.groupby('Category')['Review Rating']
      .transform(lambda x: x.fillna(x.median()))
)


In [ ]:
df['Review Rating'].isnull().sum()

## Column Name Standardization

Column names are converted into a consistent format (snake_case).

Benefits:
- Improves readability
- Reduces syntax errors
- Makes SQL and Python code easier to write
- Follows industry best practices

In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [ ]:
df.columns

## Customer Age Distribution

This helps identify which age groups make up most of the customer base and whether the business is attracting younger, middle-aged, or older customers.

In [ ]:
# create a new column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [ ]:
df[['age','age_group']].head(10)

## Standardizing Purchase Frequency

The purchase frequency values are stored as text labels such as Weekly, Monthly, and Annually.

To make the data easier to analyze, I converted these categories into approximate numbers of days. This allows future calculations and comparisons to be performed more easily.

In [ ]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

## Checking for Duplicate Information

Before building the final dataset, I wanted to check whether the `discount_applied` and `promo_code_used` columns contained the same information.

Keeping duplicate columns can increase dataset size and make analysis more confusing, so it is a good practice to identify and remove redundant data.

In [ ]:
df[['discount_applied','promo_code_used']].head(10)

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

### Observation

The comparison shows that both columns contain identical values for every record.

Since they provide the same information, keeping both columns would be unnecessary.

In [ ]:
# Dropping promo code used column

df = df.drop('promo_code_used', axis=1)

In [ ]:
df.columns

## Connecting Python to PostgreSQL

After cleaning and preparing the dataset in Python, I connected it to PostgreSQL.

This allows SQL queries to be executed on the processed data and helps combine Python-based analysis with database operations commonly used in real-world analytics projects.

In [ ]:
!pip install psycopg2-binary sqlalchemy

In [ ]:
from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL

username = "your_username"    
password =  "your_password"
host = "localhost"         
port = "5432"              
database = "customer_behavior"   

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Step 2: Load DataFrame into PostgreSQL
table_name = "customer"   
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")